## 依赖导入

本案例集以 `networkx` 建立并求解各类图优化问题（最短路径、最小生成树、最大流、图着色、TSP 等），并用 `cvxpy` 建立整数/线性规划模型求解运筹优化实例；绘图接口经 `pylab` 导入为 `plt`，`pandas` 负责 Excel/文件导出，`norm` 用于 PERT 的正态概率计算，`itertools` 提供相邻顶点配对。

In [ ]:
import itertools

import cvxpy as cp
import networkx as nx
import numpy as np
import pandas as pd
import pylab as plt
from scipy.stats import norm

## 全局绘图配置

设置全局字号 `plt.rc("font", size=16)`，使后续各网络图的节点与边标签文字统一放大、清晰可读。

In [ ]:
plt.rc("font", size=16)

## 4.1  铁路—公路联合调运的运费矩阵与购运优化

构造 7 个产地 $S_i$、15 个公路站 $A_i$、17 个铁路站 $B_i$ 共 39 个顶点。先建铁路赋权图 `G1`，用 Floyd–Warshall 求最短距离矩阵 `d1`，再按距离分段折算为单位运价矩阵 `c1`（0 元、20…60 元及每超 100 公里加收 5 元）；建公路赋权图 `G2`，由其邻接矩阵 `c2` 得到每公里 0.1 的公路费用并与 `c1` 逐元素取小合并为综合费用图 `G3`，再次 Floyd 得到运价矩阵，截取 7×15 的产—销运费块 `c5` 写入 `data/6_1.xlsx`。随后用 `cvxpy` 建立购运整数规划：以产量上限 `s`、购价 `p`、铁路运能 `b` 组合出购运总费用 `c`，决策变量为调运量 `x`、累计量 `y/z` 与 0–1 开通变量 `t`，最小化总费用加二次惩罚，约束包含产/销平衡、铁路运量递推与容量；用 CPLEX 求解并打印最优值与最优解，再把费用矩阵、解分别写入 Excel 的 Sheet2、Sheet3（源程序 anli6_1.py）。

In [ ]:
s1 = ["S" + str(i) for i in range(1, 8)]
s2 = ["A" + str(i) for i in range(1, 16)]
s3 = ["B" + str(i) for i in range(1, 18)]
L = s1 + s2 + s3
G1 = nx.Graph()
G1.add_nodes_from(L)
L1 = [
    ("B1", "B3", 450),
    ("B2", "B3", 80),
    ("B3", "B5", 1150),
    ("B5", "B8", 1100),
    ("B4", "B6", 306),
    ("B6", "B7", 195),
    ("S1", "B7", 20),
    ("S1", "B8", 202),
    ("S2", "B8", 1200),
    ("B8", "B9", 720),
    ("S3", "B9", 690),
    ("B9", "B10", 520),
    ("B10", "B12", 170),
    ("S4", "B12", 690),
    ("S5", "B11", 462),
    ("B11", "B12", 88),
    ("B12", "B14", 160),
    ("B13", "B14", 70),
    ("B14", "B15", 320),
    ("B15", "B16", 160),
    ("S6", "B16", 70),
    ("B16", "B17", 290),
    ("S7", "B17", 30),
]
G1.add_weighted_edges_from(L1)
d1 = nx.floyd_warshall_numpy(G1)
d1 = np.array(d1)
c1 = np.ones(d1.shape) * np.inf
c1[d1 == 0] = 0
c1[(d1 > 0) & (d1 <= 300)] = 20
c1[(d1 > 300) & (d1 <= 350)] = 23
c1[(d1 > 350) & (d1 <= 400)] = 26
c1[(d1 > 400) & (d1 <= 450)] = 29
c1[(d1 > 450) & (d1 <= 500)] = 32
c1[(d1 > 500) & (d1 <= 600)] = 37
c1[(d1 > 600) & (d1 <= 700)] = 44
c1[(d1 > 700) & (d1 <= 800)] = 50
c1[(d1 > 800) & (d1 <= 900)] = 55
c1[(d1 > 900) & (d1 <= 1000)] = 60
ind = (d1 > 1000) & (d1 < np.inf)
c1[ind] = 60 + 5 * np.ceil(d1[ind] / 100 - 10)

G2 = nx.Graph()
G2.add_nodes_from(L)
L2 = [
    ("A1", "A2", 104),
    ("A2", "B1", 3),
    ("A2", "A3", 301),
    ("A3", "B2", 2),
    ("A3", "A4", 750),
    ("A4", "B5", 600),
    ("A4", "A5", 606),
    ("A5", "B4", 10),
    ("A5", "A6", 194),
    ("A6", "B6", 5),
    ("A6", "A7", 205),
    ("A7", "B7", 10),
    ("S1", "A7", 31),
    ("A7", "A8", 201),
    ("A8", "B8", 12),
    ("A8", "A9", 680),
    ("A9", "B9", 42),
    ("A9", "A10", 480),
    ("A10", "B10", 70),
    ("A10", "A11", 300),
    ("A11", "B11", 10),
    ("A11", "A12", 220),
    ("A12", "B13", 10),
    ("A12", "A13", 210),
    ("A13", "B15", 62),
    ("A13", "A14", 420),
    ("S6", "A14", 110),
    ("A14", "B16", 30),
    ("A14", "A15", 500),
    ("A15", "B17", 20),
    ("S7", "A15", 20),
]
G2.add_weighted_edges_from(L2)
c2 = nx.to_numpy_matrix(G2)
c2 = np.array(c2)
c2[c2 == 0] = np.inf
c3 = np.minimum(c1, 0.1 * c2)

G3 = nx.Graph(c3)
c4 = nx.floyd_warshall_numpy(G3)
c5 = c4[:7, 7:22]
f = pd.ExcelWriter("data/6_1.xlsx")
pd.DataFrame(c5).to_excel(f, index=False)

s = np.array([800, 800, 1000, 2000, 2000, 2000, 3000])
p = np.array([160, 155, 155, 160, 155, 150, 160])
b = np.array([104, 301, 750, 606, 194, 205, 201, 680, 480, 300, 220, 210, 420, 500])
c = np.tile(p, (15, 1)).T + c5
x = cp.Variable((7, 15), integer=True)
y = cp.Variable(15, pos=True)
z = cp.Variable(15, pos=True)
t = cp.Variable(7, integer=True)
obj = cp.Minimize(cp.sum(cp.multiply(c, x)) + 0.05 * cp.sum(y**2 + y + z**2 + z))
con = [
    500 * t <= cp.sum(x, axis=1),
    cp.sum(x, axis=1) <= cp.multiply(s, t),
    cp.sum(x, axis=0) == y + z,
    y[1:] + z[:-1] == b,
    y[0] == 0,
    z[14] == 0,
    t >= 0,
    t <= 1,
    x >= 0,
]
prob = cp.Problem(obj, con)
prob.solve(solver="CPLEX")
print("最优值为：", prob.value)
print("最优解为：\n", x.value)
pd.DataFrame(c).to_excel(f, "Sheet2", index=False)
pd.DataFrame(x.value).to_excel(f, "Sheet3", index=False)
f.close()

## 4.3  立方图的两种画法

生成 3-正则的立方图 `G`（`nx.cubical_graph`）并并排绘制两幅子图：左图用默认布局直接 `nx.draw` 显示顶点标号；右图用圆形布局 `circular_layout`，把顶点 0~7 重命名为 v1~v8，并以黄色方形节点、蓝色边呈现（源程序 ex6_3.py）。

In [ ]:
G = nx.cubical_graph()
fig, (ax, ax2) = plt.subplots(1, 2)
nx.draw(G, with_labels=True, ax=ax)
s = ["v" + str(i) for i in range(1, 9)]
s = dict(zip(range(8), s))
nx.draw(
    G,
    ax=ax2,
    pos=nx.circular_layout(G),
    labels=s,
    node_color="y",
    node_shape="s",
    edge_color="b",
)
plt.show()

## 4.4  图的顶点、边与邻接表

演示 `nx.Graph` 的基本操作：用 `add_node`/`add_nodes_from` 添加顶点、`add_edge` 与 `add_weighted_edges_from` 添加无权/赋权边（其中顶点 1–2 边的权重为 0.5），最后用 `G.adj` 与 `G.adjacency()` 分别打印邻接表字典与邻接表列表（源程序 ex6_4.py）。

In [ ]:
G = nx.Graph()
G.add_node(1)
G.add_nodes_from(["A", "B"])
G.add_edge("A", "B")
G.add_edge(1, 2, weight=0.5)
e = [("A", "B", 0.3), ("B", "C", 0.9), ("A", "C", 0.5), ("C", "D", 1.2)]
G.add_weighted_edges_from(e)
print(G.adj)
print(list(G.adjacency()))

## 4.5  有向图与邻接矩阵

用 `nx.DiGraph` 按边表建立 6 个顶点的有向图（须显式 `add_nodes_from` 赋顶点），采用壳层布局 `shell_layout` 绘制并加粗标签，再用 `nx.to_numpy_matrix` 导出并打印邻接矩阵（源程序 ex6_5.py）。

In [ ]:
G = nx.DiGraph()
List = [(1, 2), (1, 3), (2, 3), (3, 2), (3, 5), (4, 2), (4, 6), (5, 2), (5, 4), (6, 5)]
G.add_nodes_from(range(1, 7))
G.add_edges_from(List)
pos = nx.shell_layout(G)
nx.draw(G, pos, with_labels=True, font_weight="bold", node_color="y")
W = nx.to_numpy_matrix(G)
print(W)
plt.show()

## 4.6  赋权图的邻接矩阵与边标签

构造含 4 顶点、5 条赋权边的无向图，用 `to_numpy_matrix` 得权重邻接矩阵 `W1`、`get_edge_attributes` 得边权字典 `W2`；以弹簧布局绘制节点并在边上标注权重，打印邻接矩阵、邻接表字典/列表及 `to_dict_of_lists` 结果，最后把邻接矩阵以整数格式保存到 `data/6_6.txt`（源程序 ex6_6.py）。

In [ ]:
G = nx.Graph()
List = [(1, 3, 10), (1, 4, 60), (2, 3, 5), (2, 4, 20), (3, 4, 1)]
G.add_nodes_from(range(1, 5))
G.add_weighted_edges_from(List)
W1 = nx.to_numpy_matrix(G)
W2 = nx.get_edge_attributes(G, "weight")
pos = nx.spring_layout(G)
nx.draw(G, pos, with_labels=True, font_weight="bold")
nx.draw_networkx_edge_labels(G, pos, font_size=13, edge_labels=W2)
print("邻接矩阵为：\n", W1)
print("邻接表字典为：\n", G.adj)
print("邻接表列表为：\n", list(G.adjacency()))
print("列表字典为：\n", nx.to_dict_of_lists(G))
np.savetxt("data/6_6.txt", W1, fmt="%d")
plt.show()

## 4.7  Dijkstra 最短路径

在含 9 个顶点的赋权有向图上用 `dijkstra_path` 与 `dijkstra_path_length` 求顶点 1 到 8 的最短路径及最小费用并打印（源程序 ex6_7.py）。

In [ ]:
G = nx.DiGraph()
List = [
    (1, 2, 6),
    (1, 3, 3),
    (1, 4, 1),
    (2, 5, 1),
    (3, 2, 2),
    (3, 4, 2),
    (4, 6, 10),
    (5, 4, 6),
    (5, 6, 4),
    (5, 7, 3),
    (5, 8, 6),
    (6, 5, 10),
    (6, 7, 2),
    (7, 8, 4),
    (9, 5, 2),
    (9, 8, 3),
]
G.add_nodes_from(range(1, 10))
G.add_weighted_edges_from(List)
path = nx.dijkstra_path(G, 1, 8, weight="weight")
d = nx.dijkstra_path_length(G, 1, 8, weight="weight")
print("最短路径为：", path)
print("最小费用为：", d)

## 4.8  Floyd 最短距离矩阵与全部最短路径

对 4 顶点赋权图用 `floyd_warshall_numpy` 求最短距离矩阵并打印，再用 `shortest_path`（`bellman-ford`）得到各顶点对之间的最短路径，按上三角逐对输出（源程序 ex6_8.py）。

In [ ]:
G = nx.Graph()
List = [(1, 3, 10), (1, 4, 60), (2, 3, 5), (2, 4, 20), (3, 4, 1)]
G.add_nodes_from(range(1, 5))
G.add_weighted_edges_from(List)
d = nx.floyd_warshall_numpy(G)
print("最短距离矩阵为：\n", d)
path = nx.shortest_path(G, weight="weight", method="bellman-ford")
for i in range(1, len(d)):
    for j in range(i + 1, len(d) + 1):
        print(f"顶点{i}到顶点{j}的最短路径为：", path[i][j])

## 4.9  由距离矩阵构造有向图并求最短路

以增量方式填出 6×6 的邻接距离矩阵 `a` 并构造赋权有向图，用 `shortest_path` 求 0 到 5 的最短路径 `p` 与长度 `d`；采用壳层布局、把顶点标为 v1~v6 并显示边权，用红色加粗弧高亮最短路径，最后 `fig.savefig` 把结果保存为 `data/6_9.png`（源程序 ex6_9.py）。

In [ ]:
a = np.zeros((6, 6))
a[0, 1:] = [15, 20, 27, 37, 54]
a[1, 2:] = [15, 20, 27, 37]
a[2, 3:] = [16, 21, 28]
a[3, 4:] = [16, 21]
a[4, 5] = 17
G = nx.DiGraph(a)
p = nx.shortest_path(G, 0, 5, weight="weight")
d = nx.shortest_path_length(G, 0, 5, weight="weight")
print("path=", p)
print("d=", d)
pos = nx.shell_layout(G)
w = nx.get_edge_attributes(G, "weight")
key = range(6)
s = ["v" + str(i + 1) for i in key]
s = dict(zip(key, s))
nx.draw(G, pos, font_weight="bold", labels=s, node_color="r")
nx.draw_networkx_edge_labels(G, pos, edge_labels=w)
path_edges = list(itertools.pairwise(p))
nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color="r", width=3)
fig = plt.figure()
fig.savefig("data/6_9.png", dpi=500)
plt.show()

## 4.10  网络中心选址（最小化最大距离）

构造 6 顶点赋权图并求最短距离矩阵 `d`，逐行取最大距离得各顶点到最远点的距离，取其中最小值作为网络半径 `mmd`，其对应顶点编号 `ind` 即网络中心，打印矩阵、最小值与中心编号（源程序 ex6_10.py）。

In [ ]:
L = [
    (1, 2, 20),
    (1, 5, 15),
    (2, 3, 20),
    (2, 4, 60),
    (2, 5, 25),
    (3, 4, 30),
    (3, 5, 18),
    (5, 6, 15),
]
G = nx.Graph()
G.add_nodes_from(np.arange(1, 7))
G.add_weighted_edges_from(L)
d = nx.floyd_warshall_numpy(G)
d = np.array(d)
md = np.max(d, axis=1)
mmd = min(md)
ind = np.argmin(md) + 1
print(d)
print("最小值为：", mmd)
print("最小值的地址为：", ind)

## 4.11  最短路问题的 0–1 规划模型

以边表初始化 6×6 对称代价矩阵 `a`（缺边置 10 万），用 `cvxpy` 建立流量守恒型 0–1 整数规划：`x[i,j]` 表示是否选弧，目标最小化选中弧的总权，约束为源点发出 1 条、汇点收入 0 条外发的净平衡、其余点进出弧数相等且 `0≤x≤1`；用 GLPK_MI 求解并打印最优值、解矩阵及最短路端点下标（源程序 ex6_11.py）。

In [ ]:
L = [
    (1, 2, 18),
    (1, 5, 15),
    (2, 3, 20),
    (2, 4, 60),
    (2, 5, 12),
    (3, 4, 30),
    (3, 5, 18),
    (4, 6, 10),
    (5, 6, 15),
]
a = np.ones((6, 6)) * 100000
for i in range(len(L)):
    a[L[i][0] - 1, L[i][1] - 1] = L[i][2]
    a[L[i][1] - 1, L[i][0] - 1] = L[i][2]
x = cp.Variable((6, 6), integer=True)
obj = cp.Minimize(cp.sum(cp.multiply(a, x)))
con = [sum(x[1, :]) == 1, sum(x[:, 1]) == 0, sum(x[:, 3]) == 1, x >= 0, x <= 1]
for i in set(range(6)) - {1, 3}:
    con.append(sum(x[i, :]) == sum(x[:, i]))
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
i, j = np.nonzero(x.value)
print("最短路径的起点：", i + 1)
print("最短路径的终点：", j + 1)

## 4.12  最小生成树及其权重

在 9 顶点赋权图上用 `minimum_spanning_tree` 求最小生成树 `T`，由邻接矩阵元素之和除以 2 得树权 `w` 并打印；在圆形布局下并排绘制原连通图与最小生成树，各自标注顶点与边权（源程序 ex6_12.py）。

In [ ]:
L = [
    (0, 1, 2),
    (0, 2, 1),
    (0, 3, 3),
    (0, 4, 4),
    (0, 5, 4),
    (0, 6, 2),
    (0, 7, 5),
    (0, 8, 4),
    (1, 2, 4),
    (1, 8, 1),
    (2, 3, 1),
    (3, 4, 1),
    (4, 5, 5),
    (5, 6, 2),
    (6, 7, 3),
    (7, 8, 5),
]
G = nx.Graph()
G.add_weighted_edges_from(L)
T = nx.minimum_spanning_tree(G)
c = nx.to_numpy_matrix(T)
print("邻接矩阵 c=\n", c)
w = c.sum() / 2
print("最小生成树的权重 W=", w)
pos = nx.circular_layout(G)
fig, (ax, ax2) = plt.subplots(1, 2)
nx.draw(G, pos, with_labels=True, font_size=13, ax=ax)
w1 = nx.get_edge_attributes(G, "weight")
nx.draw_networkx_edge_labels(G, pos, edge_labels=w1, ax=ax)
nx.draw(T, pos, with_labels=True, font_weight="bold", ax=ax2)
w2 = nx.get_edge_attributes(T, "weight")
nx.draw_networkx_edge_labels(T, pos, edge_labels=w2, ax=ax2)
plt.show()

## 4.13  旅行商问题（TSP）的整数规划

以边表构造 9×9 代价矩阵 `a`，用 `cvxpy` 建立 TSP 模型：`x[i,j]` 决定弧是否入选、`u` 为子序消除变量，目标最小化总行程，约束含起点发出弧数、每点恰入一次及 MTZ 圈消除不等式；GLPK_MI 求解后打印最优值、解矩阵与非零弧端点（源程序 ex6_13.py）。

In [ ]:
L = [
    (0, 1, 2),
    (0, 2, 1),
    (0, 3, 3),
    (0, 4, 4),
    (0, 5, 4),
    (0, 6, 2),
    (0, 7, 5),
    (0, 8, 4),
    (1, 2, 4),
    (1, 8, 1),
    (2, 3, 1),
    (3, 4, 1),
    (4, 5, 5),
    (5, 6, 2),
    (6, 7, 3),
    (7, 8, 5),
]
a = np.ones((9, 9)) * 10000
for i in range(len(L)):
    a[L[i][0], L[i][1]] = L[i][2]
    a[L[i][1], L[i][0]] = L[i][2]
x = cp.Variable((9, 9), integer=True)
u = cp.Variable(9, pos=True)
obj = cp.Minimize(cp.sum(cp.multiply(a, x)))
con = [cp.sum(x[0, :]) >= 1, u[0] == 0, u[1:] >= 1, u[1:] <= 8, x >= 0, x <= 1]
for i in range(1, 9):
    con.append(sum(x[:, i]) == 1)
for i in range(9):
    for j in range(1, 9):
        con.append(u[i] - u[j] + 9 * x[i, j] <= 8)
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
i, j = np.nonzero(x.value)
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
print("i=", i)
print("j=", j)

## 4.16  图顶点着色的整数规划

由 6 个代表队的成员集合构造相邻关系得上三角邻接矩阵 `w`，转置相加得完整矩阵并求各顶点度 `deg` 与最大度 `K`；用 `cvxpy` 建立着色模型：`x[i,k]` 表示顶点 i 用颜色 k，目标最小化颜色总数 `y`，约束每点恰一色、相邻点不同色；GLPK_MI 求解后打印所用颜色数及顶点—颜色的对应关系（源程序 ex6_16.py）。

In [ ]:
L = [
    {"张", "李", "王"},
    {"李", "赵", "刘"},
    {"张", "刘", "王"},
    {"赵", "刘", "孙"},
    {"张", "王", "孙"},
    {"李", "刘", "王"},
]
w = np.zeros((6, 6))
for i in range(5):
    for j in range(i + 1, 6):
        if len(L[i] & L[j]) >= 1:
            w[i, j] = 1
ni, nj = np.nonzero(w)
w = w + w.T
deg = w.sum(axis=1)
K = int(max(deg))
n = len(w)
x = cp.Variable((n, K + 1), integer=True)
y = cp.Variable()
obj = cp.Minimize(y)
con = [cp.sum(x, axis=1) == 1, x >= 0, x <= 1]
for i in range(n):
    con.append(y >= range(1, K + 2) @ x[i, :])
for k in range(K + 1):
    for i in range(len(ni)):
        con.append(x[ni[i], k] + x[nj[i], k] <= 1)
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
i, k = np.nonzero(x.value)
print("最优值为:", prob.value)
print("最优解为：\n", x.value)
print("顶点和颜色的对应关系如下：")
print("i=", i + 1)
print("k=", k + 1)

## 4.17  网络最大流

在 8 顶点有向图上以 `capacity` 为边权，用 `maximum_flow` 求源 1 到汇 8 的最大流值 `value` 与流分配 `flow_dict`；由流字典组装流量邻接矩阵并打印，再以壳层布局绘制容量，用红色高亮有流量的弧（源程序 ex6_17.py）。

In [ ]:
L = [
    (1, 2, 6),
    (1, 3, 4),
    (1, 4, 5),
    (2, 3, 3),
    (2, 5, 9),
    (2, 6, 9),
    (3, 4, 5),
    (3, 5, 6),
    (3, 6, 7),
    (3, 7, 3),
    (4, 3, 2),
    (4, 7, 5),
    (5, 8, 12),
    (6, 5, 8),
    (6, 8, 10),
    (7, 6, 4),
    (7, 8, 15),
]
G = nx.DiGraph()
G.add_nodes_from(range(1, 9))
G.add_weighted_edges_from(L, weight="capacity")
value, flow_dict = nx.maximum_flow(G, 1, 8)
print("最大流的流量为：", value)
print("最大流为：", flow_dict)
n = len(flow_dict)
adj_mat = np.zeros((n, n), dtype=int)
for i, adj in flow_dict.items():
    for j, weight in adj.items():
        adj_mat[i - 1, j - 1] = weight
print("最大流的邻接矩阵为：\n", adj_mat)
ni, nj = np.nonzero(adj_mat)
pos = nx.shell_layout(G)
w = nx.get_edge_attributes(G, "capacity")
nx.draw(G, pos, font_weight="bold", with_labels=True, node_color="y")
nx.draw_networkx_edge_labels(G, pos, edge_labels=w)
path_edges = list(zip(ni + 1, nj + 1))
nx.draw_networkx_edges(G, pos, edgelist=path_edges, edge_color="r", width=3)
plt.show()

## 4.18  网络最大流（图算法与规划模型）

用含转运节点的二层网络（源 s、汇 t）建立 `nx.DiGraph`，逐条 `add_edge` 设置 `capacity`，调用 `maximum_flow` 求 s 到 t 的最大流值与流分配并打印，再按 `node` 顺序组装出流量的邻接矩阵 `A`（源程序 ex6_18_1.py）。

In [ ]:
node = ["s", "A", "B", "C", "D", "E", "2b", "3b", "4b", "1", "2", "3", "4", "t"]
n = len(node)
G = nx.DiGraph()
G.add_nodes_from(node)
L = [
    ("s", "A", 4),
    ("s", "B", 3),
    ("s", "C", 3),
    ("s", "D", 2),
    ("s", "E", 4),
    ("A", "2b", 1),
    ("A", "1", 1),
    ("A", "3b", 1),
    ("A", "4b", 1),
    ("B", "2b", 1),
    ("B", "1", 1),
    ("B", "3", 1),
    ("B", "4b", 1),
    ("C", "1", 1),
    ("C", "2", 1),
    ("C", "3", 1),
    ("C", "4b", 1),
    ("D", "1", 1),
    ("D", "2", 1),
    ("D", "3b", 1),
    ("D", "4b", 1),
    ("E", "2b", 1),
    ("E", "1", 1),
    ("E", "3", 1),
    ("E", "4", 1),
    ("2b", "2", 2),
    ("3b", "3", 1),
    ("4b", "4", 2),
    ("1", "t", 5),
    ("2", "t", 4),
    ("3", "t", 4),
    ("4", "t", 3),
]
for k in range(len(L)):
    G.add_edge(L[k][0], L[k][1], capacity=L[k][2])
value, flow_dict = nx.maximum_flow(G, "s", "t")
print(value)
print(flow_dict)
A = np.zeros((n, n), dtype=int)
for i, adj in flow_dict.items():
    for j, f in adj.items():
        A[node.index(i), node.index(j)] = f
print(A)

本单元改用 `cvxpy` 整数规划建立同类的分配模型：决策变量 `x` 为 4×5 的 0–1 矩阵，列方向受容量向量 `a` 限制（`cp.sum(x, axis=0) ≤ a`），并对若干行施加组合上限，目标最大化分配总量 `cp.sum(x)`，GLPK_MI 求解后打印最优值与解（源程序 ex6_18_2.py）。

In [ ]:
a = np.array([4, 3, 3, 2, 4])
x = cp.Variable((4, 5), integer=True)
obj = cp.Maximize(cp.sum(x))
con = [
    cp.sum(x, axis=0) <= a,
    x[1, 0] + x[1, 1] + x[1, 4] <= 2,
    x[2, 0] + x[2, 3] <= 1,
    sum(x[3, :-1]) <= 2,
    x >= 0,
    x <= 1,
]
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为:", prob.value)
print("最优解为：\n", x.value)

## 4.19  最大流的最小费用

在 6 顶点有向图上同时给出容量 `capacity` 与费用 `weight`，用 `max_flow_min_cost` 求 vs 到 vt 的最大流、`cost_of_flow` 计算该最大流的最小费用并打印；再把流值按节点顺序组装为邻接矩阵，输出流入汇点的总量与邻接矩阵（源程序 ex6_19.py）。

In [ ]:
L = [
    ("vs", "v2", 5, 3),
    ("vs", "v3", 3, 6),
    ("v2", "v4", 2, 8),
    ("v3", "v2", 1, 2),
    ("v3", "v5", 4, 2),
    ("v4", "v3", 1, 1),
    ("v4", "v5", 3, 4),
    ("v4", "vt", 2, 10),
    ("v5", "vt", 5, 2),
]
G = nx.DiGraph()
for k in range(len(L)):
    G.add_edge(L[k][0], L[k][1], capacity=L[k][2], weight=L[k][3])
maxFlow = nx.max_flow_min_cost(G, "vs", "vt")
print("所求最大流为：", maxFlow)
mincost = nx.cost_of_flow(G, maxFlow)
print("最小费用为：", mincost)
node = list(G.nodes())
n = len(node)
flow_mat = np.zeros((n, n))
for i, adj in maxFlow.items():
    for j, f in adj.items():
        flow_mat[node.index(i), node.index(j)] = f
print("最大流的流量为：", sum(flow_mat[:, -1]))
print("最小费用最大流的邻接矩阵为：\n", flow_mat)

## 4.20  工程最早完工时间（CPM 线性规划）

以活动表 `L`（起点、终点、工期）建立各事件最早时间的线性规划：`x`（8 维非负变量）表示各节点时间，对每条活动约束终点时间 ≥ 起点时间加工期，目标最小化 `sum(x)`，GLPK_MI 求解后打印最优值与各节点时间（源程序 ex6_20.py）。

In [ ]:
x = cp.Variable(8, pos=True)
L = [
    (1, 2, 5),
    (1, 3, 10),
    (1, 4, 11),
    (2, 5, 4),
    (3, 4, 4),
    (3, 5, 0),
    (4, 6, 15),
    (5, 6, 21),
    (5, 7, 25),
    (5, 8, 35),
    (6, 7, 0),
    (6, 8, 20),
    (7, 8, 15),
]
obj = cp.Minimize(sum(x))
con = []
for k in range(len(L)):
    con.append(x[L[k][1] - 1] >= x[L[k][0] - 1] + L[k][2])
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为", prob.value)
print("最优解为：", x.value)

## 4.21  活动最早与最晚开工时刻

沿用 ex6_20 的最早时间模型求出 `xx`，再从汇点反向递推各节点的最晚时间 `z`；据此对每条活动计算最早开工 `es` 与最晚开工 `ls`（原程序中的 `lf/ef` 未被输出，已略去），打印最早与最晚开工时刻表（源程序 ex6_21.py）。

In [ ]:
n = 8
x = cp.Variable(n, pos=True)
z = np.zeros(n)
L = [
    (1, 2, 5),
    (1, 3, 10),
    (1, 4, 11),
    (2, 5, 4),
    (3, 4, 4),
    (3, 5, 0),
    (4, 6, 15),
    (5, 6, 21),
    (5, 7, 25),
    (5, 8, 35),
    (6, 7, 0),
    (6, 8, 20),
    (7, 8, 15),
]
obj = cp.Minimize(sum(x))
con = []
for k in range(len(L)):
    con.append(x[L[k][1] - 1] >= x[L[k][0] - 1] + L[k][2])
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为", prob.value)
print("最优解为：", x.value)
xx = x.value
z[-1] = xx[-1]
for k in range(n - 1, 0, -1):
    z[k - 1] = min([z[a[1] - 1] - a[2] for a in L if a[0] == k])
print("z:", z)
es = []
ls = []
for i in range(len(L)):
    es.append([L[i][0], L[i][1], xx[L[i][0] - 1]])
    ls.append([L[i][0], L[i][1], z[L[i][1] - 1] - L[i][2]])
print("作业最早开工时间如下：")
print(es)
print("作业最晚开工时间如下：")
print(ls)

## 4.22  关键路径的 0–1 规划

以活动表 `L` 建立 8×8 的 0–1 路径矩阵 `x`，目标最大化沿路径的活动工期之和 `fun`；约束起点只发一条弧、终点只入一条弧、中间点进出平衡，GLPK_MI 求解后打印最优值、解矩阵及关键路径端点（源程序 ex6_22.py）。

In [ ]:
L = [
    (1, 2, 5),
    (1, 3, 10),
    (1, 4, 11),
    (2, 5, 4),
    (3, 4, 4),
    (3, 5, 0),
    (4, 6, 15),
    (5, 6, 21),
    (5, 7, 25),
    (5, 8, 35),
    (6, 7, 0),
    (6, 8, 20),
    (7, 8, 15),
]
x = cp.Variable((8, 8), integer=True)
fun = 0
for i in range(len(L)):
    fun = fun + x[L[i][0] - 1, L[i][1] - 1] * L[i][2]
obj = cp.Maximize(fun)
con = [x >= 0, x <= 1]
out = [a[1] - 1 for a in L if a[0] == 1]
con.append(sum(x[0, out]) == 1)
ind = [a[0] - 1 for a in L if a[1] == 8]
con.append(sum(x[ind, 7]) == 1)
for k in range(2, 8):
    out = [a[1] - 1 for a in L if a[0] == k]
    ind = [a[0] - 1 for a in L if a[1] == k]
    con.append(sum(x[k - 1, out]) == sum(x[ind, k - 1]))
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为", prob.value)
print("最优解为：\n", x.value)
ni, nj = np.nonzero(x.value)
print("关键路径的端点为:")
print(ni + 1)
print(nj + 1)

## 4.23  工期压缩的最小费用模型

活动表 `L` 含正常工期 `L[i,2]`、最短工期 `L[i,3]`、赶工单位费用 `L[i,4]`；建立节点时间 `x` 与压缩量 `y` 的混合整数规划，目标最小化总赶工费用，约束总工期 ≤49 天、每活动终点-起点+压缩量≥正常工期、压缩量 ≤（正常-最短）工期；GLPK_MI 求解后打印最优值、`x`、`y` 及需压缩作业的两端（源程序 ex6_23.py）。

In [ ]:
L = [
    (1, 2, 5, 5, 0),
    (1, 3, 10, 8, 700),
    (1, 4, 11, 8, 400),
    (2, 5, 4, 3, 450),
    (3, 4, 4, 4, 0),
    (3, 5, 0, 0, 0),
    (4, 6, 15, 15, 0),
    (5, 6, 21, 16, 600),
    (5, 7, 25, 22, 300),
    (5, 8, 35, 30, 500),
    (6, 7, 0, 0, 0),
    (6, 8, 20, 16, 500),
    (7, 8, 15, 12, 400),
]
x = cp.Variable(8, pos=True)
y = cp.Variable((8, 8), integer=True)
fun = 0
for i in range(len(L)):
    fun = fun + y[L[i][0] - 1, L[i][1] - 1] * L[i][4]
obj = cp.Minimize(fun)
con = [x[7] - x[0] <= 49, y >= 0]
for i in range(len(L)):
    con.append(x[L[i][1] - 1] - x[L[i][0] - 1] + y[L[i][0] - 1, L[i][1] - 1] >= L[i][2])
    con.append(y[L[i][0] - 1, L[i][1] - 1] <= L[i][2] - L[i][3])
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为", prob.value)
print("x 的取值为：\n", x.value)
print("y 的取值为：\n", y.value)
ni, nj = np.nonzero(y.value)
print("压缩工期的作业为:")
print(ni + 1)
print(nj + 1)

## 4.24  工期压缩与时间参数计算

在赶工模型（同 ex6_23）目标中加入 `sum(x)` 一并最小化，求得压缩量 `yy`、节点时间 `xx` 后由汇点反推最晚时间 `z`，计算各活动的最早开工 `es` 与含压缩量的最晚开工 `ls` 并打印（源程序 ex6_24.py）。

In [ ]:
L = [
    (1, 2, 5, 5, 0),
    (1, 3, 10, 8, 700),
    (1, 4, 11, 8, 400),
    (2, 5, 4, 3, 450),
    (3, 4, 4, 4, 0),
    (3, 5, 0, 0, 0),
    (4, 6, 15, 15, 0),
    (5, 6, 21, 16, 600),
    (5, 7, 25, 22, 300),
    (5, 8, 35, 30, 500),
    (6, 7, 0, 0, 0),
    (6, 8, 20, 16, 500),
    (7, 8, 15, 12, 400),
]
n = 8
x = cp.Variable(n, pos=True)
y = cp.Variable((n, n), integer=True)
fun = 0
for i in range(len(L)):
    fun = fun + y[L[i][0] - 1, L[i][1] - 1] * L[i][4]
obj = cp.Minimize(fun + sum(x))
con = [x[7] - x[0] <= 49, y >= 0]
for i in range(len(L)):
    con.append(x[L[i][1] - 1] - x[L[i][0] - 1] + y[L[i][0] - 1, L[i][1] - 1] >= L[i][2])
    con.append(y[L[i][0] - 1, L[i][1] - 1] <= L[i][2] - L[i][3])
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为", prob.value)
print("x 的取值为：\n", x.value)
print("y 的取值为：\n", y.value)
xx = x.value
yy = y.value
z = np.zeros(n)
z[-1] = xx[-1]
for k in range(n - 1, 0, -1):
    z[k - 1] = min(
        [z[a[1] - 1] - a[2] + yy[a[0] - 1, a[1] - 1] for a in L if a[0] == k]
    )
es = []
ls = []
for i in range(len(L)):
    es.append([L[i][0], L[i][1], xx[L[i][0] - 1]])
    ls.append(
        [L[i][0], L[i][1], z[L[i][1] - 1] - L[i][2] + yy[L[i][0] - 1, L[i][1] - 1]]
    )
print(es)
print(ls)

## 4.25  PERT 计划评审技术

对含乐观/最可能/悲观三值的 PERT 网络，用 `et=(a+4m+b)/6`、`dt=(b-a)^2/36` 求各活动期望工期与方差；再用 0–1 规划（同关键路径模型）求期望工期最长的路径得期望完工时间 `f`，对关键活动的方差求和取平方根得标准差 `s`，用正态分布 `norm` 计算按期完工概率 `p` 及 95% 置信所需天数 `N`（源程序 ex6_25.py）。

In [ ]:
L = np.array(
    [
        (1, 2, 3, 5, 7),
        (1, 3, 8, 9, 16),
        (1, 4, 8, 11, 14),
        (2, 5, 3, 4, 5),
        (3, 4, 2, 4, 6),
        (3, 5, 0, 0, 0),
        (4, 6, 8, 16, 18),
        (5, 6, 18, 20, 28),
        (5, 7, 18, 25, 32),
        (5, 8, 26, 33, 52),
        (6, 7, 0, 0, 0),
        (6, 8, 11, 21, 25),
        (7, 8, 12, 15, 18),
    ]
)
et = (L[:, 2] + 4 * L[:, 3] + L[:, 4]) / 6
dt = (L[:, 4] - L[:, 2]) ** 2 / 36
n = 8
x = cp.Variable((n, n), integer=True)
m = len(L)
fun = 0
for i in range(m):
    fun = fun + x[L[i][0] - 1, L[i][1] - 1] * et[i]
obj = cp.Maximize(fun)
con = [x >= 0, x <= 1]
out = np.where(L[:, 0] == 1)[0]
con.append(cp.sum(x[0, L[out, 1] - 1]) == 1)
for k in range(2, n):
    out = np.where(L[:, 0] == k)[0]
    ind = np.where(L[:, 1] == k)[0]
    con.append(cp.sum(x[L[ind, 0] - 1, k - 1]) == cp.sum(x[k - 1, L[out, 1] - 1]))
ind = np.where(L[:, 1] == n)
con.append(cp.sum(x[L[ind, 0] - 1, n - 1]) == 1)
prob = cp.Problem(obj, con)
prob.solve(solver="GLPK_MI")
print("最优值为：", prob.value)
print("最优解为:\n", x.value)
f = prob.value
xx = x.value
s2 = 0
for i in range(m):
    s2 = s2 + xx[L[i][0] - 1, L[i][1] - 1] * dt[i]
s = np.sqrt(s2)
p = norm.cdf(52, f, s)
N = norm.ppf(0.95) * s + f
print("标准差 s:", s)
print("概率 p:", p)
print("需要天数 N：", N)